# JSON 파일 from_texts로 디스크에 저장하기
- 메타데이터와 구분해주어야 함

In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import chromadb

# DB 내 collection 만들기

## DB 주소 및 API 변수에 저장
- DB 주소 원하는대로 바꿔도 Ok
- API도 공용 API KEY로 바꾸기

In [2]:
DB_PATH = "G:/내 드라이브/DIMA5/프로젝트/Chroma DB/chroma_db"

In [3]:
MY_API = ""

## client 호출
- chromadb 객체임

In [4]:
client = chromadb.PersistentClient(path=DB_PATH)
client.heartbeat()

1744716144825528700

## collection 만들기
- chromadb 객체의 임베딩 함수는 OpenAIEmbeddingFunction
- Trit이라는 이름의 collection 만들기

In [10]:
from chromadb.utils import embedding_functions
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

embedding_function = OpenAIEmbeddingFunction(api_key = MY_API, model_name='text-embedding-3-small')

collection = client.create_collection(name="Trit", embedding_function=embedding_function)

In [11]:
# 만들어진 collection 확인하기
collection.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'data': None,
 'metadatas': [],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

# 파일 로딩 및 분할
- JSON 파일 말고 TXT 파일로 DB에 저장할 것 -> 바로 아래 셀 2개는 실행하지 마세요!
- 3.1.1. 도 설명 읽고 넘어가기
- 3.1.2.부터 실행

In [12]:
# 파일 로드
with open('../HS 코드 분류/DB 쌓기/trade_words.json', 'r', encoding='utf-8') as f:
    trade_words = json.load(f)

In [14]:
trade_words[0]

{'content': '가격위험 (Price Risk)는 시가의 변동(Market Fluctuation)에 의해서 입는 위험을 말한다. 선물거래의 헷징을 이용하거나 제조업자와 가계약을 맺고 그 계약에 의거하여 확정오퍼(Firm Offer)를 보내고 이것에 의해 매매계약을 체결하는 방법에 의해 가격위험을 회피한다.',
 'source': '무역용어'}

## json 항목 하나하나가 너무 짧음 -> 여러개 묶어주기
- 청크 갯수 줄이기 위함
- 방법 2를 채택

### 방법 1 : json 파일의 content를 여러개 묶기

In [ ]:
def group_contents_simple(data, group_size=5):
    return [
        {
            "content": "\n".join(item["content"] for item in data[i:i + group_size]),
            "metadata": {"source": "무역용"}
        }
        for i in range(0, len(data), group_size)
    ]

In [ ]:
grouped_data = group_contents_simple(trade_words, group_size=5)

In [ ]:
from langchain.schema import Document

documents = [
    Document(page_content=item["content"], metadata=item["metadata"])
    for item in grouped_data
]

### 방법 2 : 무역용어 데이터를 애초에 json 말고 txt로 통합한 뒤 chunk로 분할

In [17]:
with open('../HS 코드 분류/dataset/무역용어사전_병합본.txt', 'r', encoding='utf-8') as f:
    words = f.read()

In [18]:
# 전처리(큰따옴표 없애기 + 문단 확실히 나누기)
words = words.replace('"', '')
words = words.replace('\n', '\n\n')
words = words[9:]

In [25]:
# 청크로 나누기
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,       # 단어별로 문맥 이어지는게 아니므로 오버랩 0
    length_function=len
)

In [26]:
chunks = text_splitter.split_text(words)

In [27]:
len(chunks)

1343

# 분할된 청크 임베딩 및 DB에 저장
- 임베딩 + 저장 한꺼번에 하는 기능은 chromadb가 아닌 Langchain의 Chroma가 진행 => Chroma.from_texts()
- Langchain의 Chroma는 임베딩 함수로 OpenAIEmbeddings 사용
- 메타데이터도 저장하기 위해 따로 함수로 만들어두기

In [33]:
# 랭체인 크로마를 쓰기 위해서는 반드시 필요
embeddings = OpenAIEmbeddings(api_key=MY_API, model='text-embedding-3-small')

In [31]:
# 메타데이터의 갯수는 청크(content)의 갯수와 동일해야 함
metadatas = [{"source": "무역용어"}] * len(chunks)

In [34]:
# 'Trit' collection에 저장하기
db = Chroma.from_texts(
    texts = chunks,
    embedding=embeddings,
    metadatas=metadatas,
    persist_directory=DB_PATH,
    collection_name='Trit'
)

In [35]:
# 내가 사용할 db (Langchain의 Chroma 객체) 첫번째 항목 살펴기기
db.get(limit=1)

{'ids': ['cbe530af-b381-41ef-bf89-ed58a9b62ff1'],
 'embeddings': None,
 'documents': ['가격위험 (Price Risk)는 시가의 변동(Market Fluctuation)에 의해서 입는 위험을 말한다. 선물거래의 헷징을 이용하거나 제조업자와 가계약을 맺고 그 계약에 의거하여 확정오퍼(Firm Offer)를 보내고 이것에 의해 매매계약을 체결하는 방법에 의해 가격위험을 회피한다.\n\n가격인하 (Abatement)는 무역거래에서는 채무나 손해배상금을 경감하거나 세금의 미지급분의 일부 또는 전부를 취소하는 것을 말한다.'],
 'uris': None,
 'data': None,
 'metadatas': [{'source': '무역용어'}],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

# LLM 요청

## retriever 지정: 검색기로 내가 만든 db 사용

In [36]:
retriever = db.as_retriever()

## 체인 만들기

In [37]:
from langchain_openai import ChatOpenAI                                # openai API 사용
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain import hub
from langchain.embeddings import OpenAIEmbeddings

In [38]:
# rag_chain 안에 들어갈 llm과 prompt 지정
llm = ChatOpenAI(
    temperature=0,
    openai_api_key=MY_API,
    max_tokens=3000,
    model_name="gpt-3.5-turbo",
    request_timeout=120
)

prompt = hub.pull("rlm/rag-prompt")

# rag_chain 만들기
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

C:\Users\user\anaconda3\Lib\site-packages\langsmith\client.py:277: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


# 테스트

In [39]:
answer = rag_chain.invoke("가격증감약관이 뭐야?")
print(answer)

가격증감약관은 물가나 환율의 변동에 따라 거래계약가격 등을 변경하는 조항을 미리 계약에 규정하는 것을 말합니다. 이는 CIF 계약성립 이후에도 원재료비, 운임, 보험료, 환율의 변동이 있는 경우 계약가격의 변경을 인정하는 약관입니다. 가격증감약관은 'Fluctuation Clause', 'Escalation Clause'으로도 불립니다.


In [40]:
answer = rag_chain.invoke("FCA가 뭔지 알려줄래?")
print(answer)

FCA는 Free Carrier의 약자로, 매도인이 지정된 장소에서 매수인이 지명한 운송인의 관리 하에 수출통관된 물품을 인도하는 조건을 의미합니다. FCA 조건은 운송기술과 국제복합운송의 발달에 따라 개편된 무역조건으로, 철도운송, 해상운송, 내수로운송, 복합운송 등에 적용됩니다. FCA 조건의 도입 배경은 복합운송 전용의 FRC와 FOR-FOT 및 FOA를 통합하여 발전된 것입니다.


In [41]:
answer = rag_chain.invoke("인코텀스는 무엇인가요?")
print(answer)

인코텀즈는 국제상업회의소(ICC)가 제정한 정형거래조건을 말하며, 무역거래의 계약당사자들 사이에 발생할 수 있는 무역분쟁을 예방하고 해석의 차이로 인한 불확실성을 최소화하기 위한 규칙입니다. 인코텀즈는 무역조건의 해석에 관한 국제규칙으로, 무역업자 간에 발생하는 오해나 분쟁을 방지하고 국제무역거래의 관습과 용어를 통일하기 위해 사용됩니다.
